# Clase 6: Visualización Táctica

Un gráfico vale más que mil tablas SQL. Veremos cómo visualizar la variabilidad y el rendimiento usando Python.

## Objetivo
- Crear gráficos que muestren historias reales, no solo "resúmenes".
- **Scatter Plots:** Relación entre dos variables (ej. Salario vs HR).
- **Boxplots:** Ver consistencia y valores atípicos (outliers).

---

## Carga de Datos

In [ ]:
# Configuración de Kaggle 
import os
from google.colab import files
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if not os.path.exists("/root/.kaggle/kaggle.json"):
    if not os.path.exists("kaggle.json"):
        print("Por favor, sube tu archivo kaggle.json:")
        # files.upload() # Descomentar en Colab
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Descargar datos si no existen
if not os.path.exists("baseball_data"):
    print("Descargando dataset de Baseball...")
    !kaggle datasets download -d open-source-sports/baseball-databank
    !unzip -q baseball-databank.zip -d baseball_data
    print("Dataset descargado.")

# Carga de tablas
try:
    batting = pd.read_csv('baseball_data/core/Batting.csv')
    salaries = pd.read_csv('baseball_data/core/Salaries.csv')
    teams = pd.read_csv('baseball_data/core/Teams.csv')
except FileNotFoundError:
    # Fallback si la estructura es diferente
    batting = pd.read_csv('baseball_data/Batting.csv')
    salaries = pd.read_csv('baseball_data/Salaries.csv')
    teams = pd.read_csv('baseball_data/Teams.csv')

print("Tablas cargadas: Batting, Salaries, Teams")

# Merge inicial para ejemplos existentes
df_merged = pd.merge(batting, salaries, on=['yearID', 'teamID', 'playerID'])


## 1. Scatter Plot: ¿El dinero compra Home Runs?
Vamos a graficar Salario vs HR para el año 2010.

In [ ]:
data_2010 = df_merged[df_merged['yearID'] == 2010]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=data_2010, x='salary', y='HR', alpha=0.6)
plt.title('Relación Salario vs Home Runs (2010)')
plt.xlabel('Salario (USD)')
plt.ylabel('Home Runs')
plt.grid(True)
plt.show()

**Reflexión:** ¿Ves una linea clara? ¿O hay jugadores baratos que hacen muchos HR (zona superior izquierda)?

## 2. Boxplot: Distribución de HR por Equipo
El promedio miente. El Boxplot nos muestra qué tan dispersos están los datos de cada equipo.

In [ ]:
top_equipos = ['NYA', 'BOS', 'LAN', 'CHA']
data_top = data_2010[data_2010['teamID'].isin(top_equipos)]

plt.figure(figsize=(10, 6))
sns.boxplot(data=data_top, x='teamID', y='HR')
plt.title('Distribución de Home Runs por Equipo (2010)')
plt.show()

### Interpretación
- La **línea central** de la caja es la **Mediana**.
- La **caja** encierra el 50% de los jugadores.
- Los **puntos** fuera de los "bigotes" son valores atípicos (estrellas o casos raros).

## 3. Baseball: Moneyball (Salarios vs Victorias)
¿Gastar más garantiza ganar más? Comparemos la nómina total del equipo con sus victorias.


In [ ]:
# Preparar datos: Agrupar salarios por equipo y año, y unir con victorias (Teams)
team_salaries = salaries.groupby(['yearID', 'teamID'])['salary'].sum().reset_index()
team_stats = pd.merge(team_salaries, teams, on=['yearID', 'teamID'])

# Filtramos para el año 2010
moneyball_2010 = team_stats[team_stats['yearID'] == 2010]

plt.figure(figsize=(12, 8))
sns.scatterplot(data=moneyball_2010, x='salary', y='W', s=100, hue='W', palette='viridis')

# Etiquetas para algunos equipos
for line in range(0, moneyball_2010.shape[0]):
    if moneyball_2010.W.iloc[line] > 90 or moneyball_2010.salary.iloc[line] < 6e7:
         plt.text(moneyball_2010.salary.iloc[line]+0.2, moneyball_2010.W.iloc[line], 
                  moneyball_2010.teamID.iloc[line], horizontalalignment='left', size='small', color='black')

plt.title('Moneyball 2010: Costo por Victoria')
plt.xlabel('Nómina Total (USD)')
plt.ylabel('Victorias (W)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 4. Basketball: Shot Charts y Heatmaps
Utilizaremos datos sintéticos para simular la carta de tiro de un jugador. 
Crearemos coordenadas (X, Y) simulando una media cancha de baloncesto.


In [ ]:
# Generación de Datos Sintéticos de Baloncesto
np.random.seed(42)
n_shots = 300

# Coordenadas: X (-250 a 250), Y (-50 a 420) - Unidades típicas en pies x10
shots_df = pd.DataFrame({
    'x': np.random.normal(0, 100, n_shots),  # Concentrados en el centro
    'y': np.random.normal(150, 100, n_shots), # Distancia al aro
    'outcome': np.random.choice(['Made', 'Missed'], n_shots, p=[0.45, 0.55])
})

# Limitar a dimensiones de cancha
shots_df = shots_df[(shots_df['x'] > -250) & (shots_df['x'] < 250) & (shots_df['y'] < 400) & (shots_df['y'] > 0)]
print(f'Disparos simulados: {len(shots_df)}')


In [ ]:
# Shot Chart Simple
plt.figure(figsize=(10, 9))
sns.scatterplot(data=shots_df, x='x', y='y', hue='outcome', palette={'Made': 'green', 'Missed': 'red'}, style='outcome')
plt.title('Shot Chart: Distribución de Tiros')
plt.xlim(-250, 250)
plt.ylim(0, 420)
# Dibujar aro (aprox)
circle = plt.Circle((0, 0), 7.5, color='orange', fill=False)
plt.gca().add_patch(circle)
plt.show()


In [ ]:
# Heatmap de Densidad de Tiros
plt.figure(figsize=(10, 8))
sns.kdeplot(data=shots_df, x='x', y='y', fill=True, cmap='rocket_r', thres=0.05, levels=15)
plt.title('Heatmap: Zonas de Calor Ofensivo')
plt.xlim(-250, 250)
plt.ylim(0, 420)
plt.show()


## 5. Soccer: Radares de MVP y Tracking
Analizaremos el rendimiento de jugadores usando gráficos de radar (Spider Plots) y mapas de calor de posición.


In [ ]:
# Datos Sintéticos de Jugadores de Fútbol
players_data = pd.DataFrame({
    'Player': ['Messi', 'Ronaldo', 'Mbappe'],
    'Pace': [85, 90, 97],
    'Shooting': [92, 93, 89],
    'Passing': [91, 82, 80],
    'Dribbling': [95, 85, 92],
    'Physical': [70, 85, 78]
})

print(players_data)


In [ ]:
# Gráfico de Radar (Complejo en Matplotlib, pero muy útil)
from math import pi

def create_radar(player_name, data, color):
    categories = list(data.columns[1:])
    N = len(categories)
    
    # Valores del jugador
    values = data[data['Player'] == player_name].iloc[0, 1:].values.flatten().tolist()
    values += values[:1] # Cerrar el ciclo
    
    # Angulos
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    
    ax = plt.subplot(111, polar=True)
    plt.xticks(angles[:-1], categories)
    ax.plot(angles, values, linewidth=1, linestyle='solid', label=player_name, color=color)
    ax.fill(angles, values, color, alpha=0.25)

plt.figure(figsize=(8, 8))
create_radar('Messi', players_data, 'b')
create_radar('Ronaldo', players_data, 'r')
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.title('Comparación de Estrellas')
plt.show()


---
## Tarea: Desafíos de Visualización

### Ejercicio 1: Evolución de Home Runs
Usa la tabla `Teams` para graficar el **promedio de HR por año** desde 1920 hasta 2010. ¿Puedes identificar la 'Era de los Esteroides' (años 90-2000)?

### Ejercicio 2: Boxplot de Salarios
Crea un Boxplot que compare la distribución de salarios (`salary`) de los **New York Yankees (NYA)**, **Boston Red Sox (BOS)** y **Oakland Athletics (OAK)** en el año 2005.

### Ejercicio 3: Mapa de Calor de Fútbol (Reto)
Genera un DataFrame sintético de 500 puntos (x, y) que simule a un **Lateral Derecho** (concentrado en la banda derecha del campo, digamos x > 50, y entre 0 y 20). Grafica su `kdeplot`.


In [ ]:
# Tu solución al Ejercicio 1 aquí (Line Plot)


In [ ]:
# Tu solución al Ejercicio 2 aquí (Boxplot)


In [ ]:
# Tu solución al Ejercicio 3 aquí (Heatmap)
